<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

Code based on https://github.com/pytorch/examples/blob/master/mnist/main.py

In this exercise, we are using high-level abstractions from torch.nn like nn.Linear.
Note: during the next lab session we will go one level deeper and implement more things
with bare hands.

Tasks:

 1. Read the code.

 2. Check that the given implementation reaches 95% test accuracy for architecture input-128-128-10 after few epochs.

 3. Add the option to use SGD with momentum instead of ADAM.

 4. Experiment with different learning rates. Use the provided TrainingVisualizer
 to plot the learning curves and gradient-to-weight ratios. Compare visualizations
 for different learning rates for both ADAM and SGD with momentum.

 5. Parameterize the constructor by a list of sizes of hidden layers of the MLP.
 Note that this requires creating a list of layers as an attribute of the Net class,
 and one can't use a standard Python list containing nn.Modules (why?).
 Check torch.nn.ModuleList.

If you run this notebook locally then you may need to install some packages.
It may be achieved by adding the following code cell to the notebook and running it:
```
!pip install torch torchvision plotly ipywidgets
```

This notebook will work faster on a Colab GPU (or TPU).
However, start with tests just on CPU, and try to end your GPU sessions as soon as you stop needing them (otherwise you may use all your free GPU time for this week).

In [1]:
import math
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import plotly.graph_objects as go
from torch import Tensor
from torchvision import datasets, transforms

# Allow custom (non-ipywidget) widgets.
if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()

# For reproducibility.
torch.manual_seed(1)

## Tensors
In ML, *tensor* is just a short name for an n-dimensional array.

In other areas like physics, it might mean a multilinear operation that can be represented, after a choice of basis, by an ndarray; this is where we would see all the extra stuff (like covariance/contravariance), like for example on the Wikipedia article for "tensor". For us, it's usually just a bunch of numbers.

torch.Tensor is can be thought of as a wrapper around numpy ndarray, but:
* the interface is slightly different (like `np.expand_dims(x, dim=n-1)` vs `torch.unsqueeze(x, axis=-1)`);
* the data can be stored in GPU memory (or others);
* it can be part of a computation graph: a computed Tensor has pointers to other Tensors it was computed from, and to functions computing partial derivatives.

All the operations defined for a torch.Tensor (`@`, `.arctan`, `.unsqueeze`, and a hundred others) can also run on a GPU and other backends.

In [2]:
x_np = np.array([[0, 1, 2], [3, 4, 5]])
x = torch.tensor(
    [[0, 1, 2], [3, 4, 5]]
)  # Also: torch.rand, ones, zeros, rand_like, ...
display(x_np, x)

array([[0, 1, 2],
       [3, 4, 5]])

tensor([[0, 1, 2],
        [3, 4, 5]])

In [3]:
torch.from_numpy(x_np).equal(x), np.array_equal(x_np, x.numpy())

(True, True)

In [4]:
x.shape, x.dtype, x.device, x.requires_grad

(torch.Size([2, 3]), torch.int64, device(type='cpu'), False)

Most operations have an in-place variant with the same name, but with an underscore.
This forgets the original value, making the computation of gradients impossible, so most of the time we don't use in-place operations.

In [5]:
x.add_(2)
display(x)

tensor([[2, 3, 4],
        [5, 6, 7]])

In [6]:
x.add(2)
display(x)
x.add_(2)
display(x)

tensor([[2, 3, 4],
        [5, 6, 7]])

tensor([[4, 5, 6],
        [7, 8, 9]])

In [52]:
tens = torch.tensor([[1.0,2.0,3.0], [4.0,5.0,6.0]], requires_grad=True)
tens1 = torch.tensor([[1.0,2.0,3.0], [4.0,5.0,6.0]], requires_grad=True)

tens2 = tens + tens1
y0 = tens2 + 1
y1 = (y0**2 - tens1).mean()
y2 = y0.sum()

print(y1)

y1.backward()
y2.backward()
print(tens.grad)



tensor(72.1667, grad_fn=<MeanBackward0>)
tensor([[2.0000, 2.6667, 3.3333],
        [4.0000, 4.6667, 5.3333]])


In [54]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
W = torch.tensor([[1., 2., 3.], [4., 5., 6.]], requires_grad=True)
b = torch.tensor([[1., 2., 3.], [4., 5., 6.]], requires_grad=True)

y0 = x * W + b + 1
loss = (y0**2 - b).mean()
loss.backward()
W.grad


tensor([[ 1.0000,  4.6667, 13.0000],
        [28.0000, 51.6667, 86.0000]])

## Autograd
Operations on torch tensors remember the computation graph.
This allows to compute the gradient of e.g. a loss node over all other nodes with loss.backward().

In [7]:
x = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], requires_grad=True)
y = torch.sigmoid(x)
loss = ((y - 1) ** 2).mean()
print("x", x.grad_fn, ", y", y.grad_fn, ", loss", loss.grad_fn)
print(loss.grad_fn.next_functions)  # type: ignore
print(loss.grad_fn.next_functions[0][0].next_functions)  # type: ignore

x None , y <SigmoidBackward0 object at 0x7ff3adefb820> , loss <MeanBackward0 object at 0x7ff3adefbac0>
((<PowBackward0 object at 0x7ff3cf17ac80>, 0),)
((<SubBackward0 object at 0x7ff3ade78370>, 0),)


The method Tensor.backward():
* for each node with `requires_grad`: computes the gradients from each .grad_fn and propagates it using the chain rule.
* for each leaf node (`requires_grad` with no predecessors that `requires_grad`): accumulates the gradient into the tensor’s `.grad` attribute.



In [8]:
loss.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

We say *accumulates* not *stores*, because if .grad already exists, `backward()` adds to it, instead of replacing it.

In [9]:
loss2 = (x * 0).mean()
loss2.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

During (for example) evaluation, you don't need to store the computational graph, so you can disable it:

In [10]:
with torch.no_grad():
    y = x * x

print(y.grad_fn)

None


In some situations you may need more fine-grained control.
You can detach a tensor from the graph, to treat it like a constant tensor.

In [11]:
y = x * x
print(y.grad_fn, y.requires_grad)
y = y.detach()  # Not in-place, returns a new tensor with the same data.
print(y.grad_fn, y.requires_grad)

<MulBackward0 object at 0x7ff3ade37460> True
None False


## Modules
A torch **module** (not *model*) is essentially just an object with a `forward()` method, usually `forward(x: Tensor) -> Tensor`.
A network (model) is built as a large module that contains and calls smaller modules, which themselves contain even smaller modules.

Basically any callable could be a module, but subclassing `nn.Module` adds features like:
* automatically registering and enumerating submodules and trainable parameters,
* serializing and storing parameters into files,
* adding hooks in the middle of existing models.

In [12]:
class MyLinear(nn.Module):
    """
    This is essentially the same as nn.Linear.

    Compare: https://github.com/pytorch/pytorch/blob/v2.9.0/torch/nn/modules/linear.py#L53
    """

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty((out_features, in_features)))
        self.bias = nn.Parameter(torch.empty(out_features))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # The initialization used by PyTorch by default, including non-zero biases.
        bound = 1 / math.sqrt(self.in_features)
        nn.init.uniform_(self.weight, -bound, bound)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, input: Tensor) -> Tensor:
        """
        Input shape: (B*, in_features).
        Output shape: (B*, out_features).
        """
        return input @ self.weight.T + self.bias


We define the module behaviour in the **`forward()`** method, not in the special `__call__()`.<br>
However, we then use the module directly as a callable, `module()`.<br>
This will call `module.forward()`, but can also calls hooks, if any are defined.

In [55]:
module = MyLinear(2, 3)
x = torch.rand((7, 5, 2))
y = module(x)
y.shape

torch.Size([7, 5, 3])

The class **nn.Parameter** just wraps a tensor to indicate that this is a trainable parameter, and changes the default `requires_grad`.<br>
When an nn.Parameter is assigned to a field of an nn.Module, it automatically registers it,
so that all parameters of this modules and its submodules can be easily accessed.

In [14]:
for i, (name, param) in enumerate(module.named_parameters()):
    print(f'{i}. "{name}":\t{param}\n')

0. "weight":	Parameter containing:
tensor([[ 0.3643, -0.3121],
        [-0.1371,  0.3319],
        [-0.6657,  0.4241]], requires_grad=True)

1. "bias":	Parameter containing:
tensor([-0.1455,  0.3597,  0.0983], requires_grad=True)



## A basic Net

In [15]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, 28, 28).
        Output: class probabilities (after softmax), shape (B, 10).
        """
        x = torch.flatten(x, start_dim=1)  # (B, 728)
        x = self.fc1(x)  # (B, 128)
        x = F.relu(x)
        x = self.fc2(x)  # (B, 128)
        x = F.relu(x)
        x = self.fc3(x)  # (B, 10)
        output = F.log_softmax(x, dim=1)
        return output

Assigning to a nn.Module to a field of nn.Module automatically registers it as a submodule.<br>
You can iterate over all submodules (recursively) with `.named_modules()` or over direct children only with `.named_children()`.

In [16]:
net = Net()
for i, (name, submodule) in enumerate(net.named_modules()):
    print(f'{i}. "{name}":\t{submodule}')

0. "":	Net(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)
1. "fc1":	Linear(in_features=784, out_features=128, bias=True)
2. "fc2":	Linear(in_features=128, out_features=128, bias=True)
3. "fc3":	Linear(in_features=128, out_features=10, bias=True)


The `.train()` and `.eval()` methods set the `training` flag of a module and all its submodules.<br>
This changes the behaviour of certain modules like BatchNorm and Dropout (see next labs).

Forgetting to switch the training mode is a common bug that can cause weird behaviour.

In [17]:
net.train()
print(net.fc1.training)
net.eval()
print(net.fc1.training)

True
False


Note that operations on a module (`.train()`, `.eval()`, `.to(dtype)`, `.to(device)`) modify it in-place.<br>
Operations on a tensor without a trailing underscore (including `.to(dtype)`, `.to(device)`) return a new tensor and leave the original unchanged


## Train and test code

In [18]:
# @title <small>Helper class: TrainingVisualizer</small>


class TrainingVisualizer:
    def __init__(self, log_interval: int = 10):
        self.log_interval = log_interval
        self.train_loss_fig = self.init_line_plot(
            title="Training loss", xaxis_title="Step"
        )
        self.grad_to_weight_fig = self.init_line_plot(
            title="Gradient standard deviation to weight standard deviation ratio at 1st layer",
            xaxis_title="Step",
            yaxis_title="Gradient to weight ratio (log scale)",
            yaxis_type="log",
        )
        self.test_acc_fig = self.init_line_plot(
            title="Test accuracy", x=[], xaxis_title="Epoch", mode="lines+markers"
        )

        # Parameters related to current tracked model and its training
        self.first_linear_layer = None
        self.lr = None
        self.trace_idx = -1

    def init_line_plot(
        self,
        title: str,
        x=None,
        xaxis_title: str | None = None,
        yaxis_title: str | None = None,
        yaxis_type: str = "linear",
        mode: str = "lines",
    ) -> go.FigureWidget:
        fig = go.Figure()
        fig.update_layout(
            title=title,
            title_x=0.5,
            xaxis_title=xaxis_title,
            yaxis_title=yaxis_title,
            height=400,
            width=1500,
            margin=dict(b=10, t=60),
        )
        fig.update_yaxes(type=yaxis_type)
        # We cannot add new traces dynamically because Colab has a problem with widgets
        # from plotly (traces added dynamically are rendered twice).
        # As an ugly workaround we create a lot of empty traces and update them later
        # with actual data. Empty traces are not plotted.
        for _ in range(25):
            fig.add_trace(go.Scatter(x=x, y=[], showlegend=True, mode=mode))

        fig_widget = go.FigureWidget(fig)
        display(fig_widget)
        return fig_widget

    def track_model(
        self, model: torch.nn.Module, optimizer: torch.optim.Optimizer, lr: float
    ) -> None:
        """
        Start tracking training metrics for a new model.
        """

        for field in model.__dict__["_modules"].values():
            if isinstance(field, nn.Linear):
                self.first_linear_layer = field
                break
            elif isinstance(field, nn.ModuleList):
                self.first_linear_layer = field[0]
                break

        self.lr = lr
        self.trace_idx += 1

        optim_name = type(optimizer).__name__
        self.train_loss_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.grad_to_weight_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.test_acc_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"

    def plot_gradients_and_loss(self, batch_idx: int, loss: float) -> None:
        if batch_idx % self.log_interval == 0:
            self.train_loss_fig.data[self.trace_idx].y += (loss,)

            layer = self.first_linear_layer
            grad_to_weight_ratio = (
                self.lr * layer.weight.grad.std() / layer.weight.std()
            ).item()

            self.grad_to_weight_fig.data[self.trace_idx].y += (grad_to_weight_ratio,)

    def plot_accuracy(self, epoch: int, accuracy: float) -> None:
        self.test_acc_fig.data[self.trace_idx].x += (epoch,)
        self.test_acc_fig.data[self.trace_idx].y += (accuracy,)

In [56]:
def train_epoch(
    model: torch.nn.Module,
    device: torch.device,
    train_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    log_interval: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to the device (e.g. GPU #0) where model parameters are.
        data, target = data.to(device), target.to(device)

        # Since `backward()` will accumuate gradients, remember to reset them.
        optimizer.zero_grad()

        # Forward pass.
        output = model(data)  # Logs of class probabilites, shape (B, 10)
        loss = F.nll_loss(output, target)  # Shape (1,)

        # Backward pass.
        loss.backward()

        visualizer.plot_gradients_and_loss(batch_idx, loss.item())

        # For each parameter, updates the tensor's value using its .grad.
        optimizer.step()

        if batch_idx % log_interval == 0:
            if verbose:
                done, total = batch_idx * len(data), len(train_loader.dataset)
                print(
                    f"Train Epoch: {epoch} [{done}/{total} images ({done / total:.0%})]\t"
                    + f"Loss: {loss.item():.6f}"
                )

In [57]:
def test(
    model: torch.nn.Module,
    device: torch.device,
    test_loader: torch.utils.data.DataLoader,
    epoch: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.eval()
    test_loss = 0
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            # Accumulate sum of dataitem losses and only divide by n_total at the end.
            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            n_correct += (pred == target).sum().item()
            n_total += len(target)

    test_loss /= n_total

    if verbose:
        print(
            f"\nTest loss: {test_loss:.4f}, Accuracy: {n_correct}/{n_total} ({n_correct / n_total:.0%})\n"
        )
    visualizer.plot_accuracy(epoch, 100.0 * n_correct / n_total)

## Hyperparameters

In [21]:
# Training uses more memory than test (due to gradient computations),
# so we can set test_batch_size to be larger.
epochs = 5
batch_size = 256
test_batch_size = 1000
lr = 1e-2
log_interval = 10

# Check for CUDA / TPU (Google) / MPS (Apple) / XPU (Intel) / ... accelerator.
# (TPU as of 2026 gives spurious warnings about lss.item() accessing .grad - ignore them).
device = (
    torch.accelerator.current_accelerator(check_available=True)  #
    or torch.device("cpu")
)
use_accel = device != torch.device("cpu")
print(use_accel, device)

False cpu


## Dataset and dataloader
Any sequence of dataitems (and even some iterables) can be used as a ***dataset***.<br>
A dataitem can be for example an (input image, output label) pair.<br>
Torch datasets usually allow passing a transformation: it is then applied whenever you retrieve an item with `dataset[i]`.

A ***dataloader*** manages parallel loading (it can create worker threads that call `dataset[i]` in parallel), batching, shuffling, sometimes moving data between devices.

In [22]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=(0.1307,), std=(0.3081,))]
)
train_dataset = datasets.MNIST(
    "../data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST("../data", train=False, transform=transform)
len(train_dataset), len(test_dataset)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.65MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]


(60000, 10000)

In [23]:
kwargs = {"num_workers": 1, "pin_memory": True} if use_accel else {}
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, **kwargs
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=test_batch_size, **kwargs
)

## Running it all

In [24]:
model = Net().to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c369b48c-e32b-459b-ad26-be9d4dc4ccee',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '621e52d0-e122-4f9b-b5bc-eccef6ee9aa1',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '9a3e4c65-fb98-403d-b23f-d50b46e7b905',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8b9d7ed6-421e-4699-8f9c-25fc3346d292',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6d16879a-5c5b-45b7-bb3f-362ba0e2893e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '1fc8dd08-d229-4573-ad0d-a66106356ab8',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6a9c2cdf-9ae9-43c3-bc6d-34eaf9c86ae9',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5e5ec594-d3ae-4551-976a-0f693dbfd68d',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c26838f1-acc1-4dde-b297-4c3458280e24',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'cf5ba289-b15d-4096-8d3c-4451efa32494',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '00dd4a5c-ab1f-43e3-879e-e09084ca4d57',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8aa5d0db-985d-41e6-8073-caf132ebccbe',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '23511b4e-93c2-4b30-970e-6923cc5b818e',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2e6a344d-a5f5-4b52-bb22-94e8eccc710f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.314002
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 0.542665
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 0.285346
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 0.294533
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.258572
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.301675
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.362897
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.251745
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.151805
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.172183
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.143477
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.272512
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.205603
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.225461
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.167229
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.131399
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.119783
Train Epoch: 1 [43520/60000 images (73%)]

My experiments


In [68]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 128)
        self.fc5 = nn.Linear(128, 10)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.4)
        self.drop2 = nn.Dropout(0.3)
        self.drop3 = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1) # (784,)
        x = F.relu(self.fc1(x)) # (784, 512)
        x = self.drop1(F.relu(self.bn1(self.fc2(x)))) # (512, 256)
        x = self.drop2(F.relu(self.fc3(x))) # (256, 128)
        x = self.drop3(F.relu(self.bn2(self.fc4(x)))) # (128, 128)
        x = self.fc5(x) # (128, 10)
        return F.log_softmax(x, dim=1)


In [70]:
model = Net2().to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '42debc54-b955-4dbb-b771-e805396e3199',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'eac2d911-9cc1-422e-97e6-b110764f00ec',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'f8a8e10c-de2d-48f1-98b6-9471e67afbad',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b416f4f0-87ae-4fb3-9173-588f60d6c2f0',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2a84a60c-6626-4bf9-9f31-85dce05d78e0',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5bc22685-b6ee-474f-84f8-6498af1ea76d',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '0ccedd59-3a6e-4ae0-bd2f-24c9a6b3895b',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '777ff581-e8d1-4794-92ce-b356e5ac640c',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4443e0d9-7003-4f0e-a151-8d4b18863e44',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '889b9b85-6266-4bcc-afd2-e57b8837efc7',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2af1ac22-bfbe-404b-b78d-24d281cbf1e5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5ec06eb6-9d75-43e2-a784-e2ab2629c92f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '1a12de6b-9277-412c-80c5-6280f5e4a9cb',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '90c63e15-069a-4f3a-8d9f-c1b6f0da196d',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.380615
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 0.752483
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 0.416565
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 0.397091
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.332250
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.331634
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.244591
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.213131
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.412407
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.321880
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.242672
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.235366
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.320686
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.274486
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.224698
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.271931
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.234656
Train Epoch: 1 [43520/60000 images (73%)]

In [71]:
model = Net2().to(device)

optimizer2 = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=epochs)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer2,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    sched.step()
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd63851d5-04b3-49bf-bd23-07980977d890',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7f7cf344-b8e2-421c-b77e-8812f692ab16',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'efe9fb93-72ab-40fb-acaf-3ce2fb26e174',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'fe6ab1ca-16f9-48bb-b279-19699e776b87',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '93881370-84ef-4712-8a60-7ebcddd0b6b2',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '707f4067-daee-4ec3-9ed9-05a2fa5a91df',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '279f2e9d-0795-458f-906a-95ed388b91ac',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'db438c1e-5916-4eef-a273-23844fb83a0d',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '36444694-898f-4a9f-839c-49fcec4912d9',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '40509482-d596-476d-987f-131ace705481',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '3a215439-1886-4852-a0e8-6c03c3092abd',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b0d611b0-7b64-46c7-99bf-0d933775eba0',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '18f45b6c-9b80-4096-b0a8-70f18d246e51',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7c99960f-58d4-4134-8f0a-4225a821c07f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.399461
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 1.410149
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 1.011954
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 0.760710
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.572038
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.398512
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.351144
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.269840
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.245226
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.279877
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.325983
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.270163
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.181172
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.296390
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.227665
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.217863
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.200105
Train Epoch: 1 [43520/60000 images (73%)]

In [72]:
class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 128)
        self.fc5 = nn.Linear(128, 10)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.4)
        self.drop2 = nn.Dropout(0.3)
        self.drop3 = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1) # (784,)
        x = F.silu(self.fc1(x)) # (784, 512)
        x = self.drop1(F.silu(self.bn1(self.fc2(x)))) # (512, 256)
        x = self.drop2(F.silu(self.fc3(x))) # (256, 128)
        x = self.drop3(F.silu(self.bn2(self.fc4(x)))) # (128, 128)
        x = self.fc5(x) # (128, 10)
        return F.log_softmax(x, dim=1)


In [73]:
model = Net3().to(device)

optimizer2 = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=epochs)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer2,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    sched.step()
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'efc71cbb-02bb-4901-94c4-dcea98667609',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '27c84fca-ce7f-4d2c-ab51-dc4be40856b1',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'ad9de8e8-d80f-4223-a8ce-0b2ba4822152',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '1b8e7136-5d19-4289-bdd6-8a7ef3e619fd',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '59b03397-e4b7-451d-a72d-136c476335d4',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '482ae636-2316-48ea-a47a-1f7fb8c7c0e8',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '02131f57-f972-47f4-b5c0-b820934fd397',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '43851aa4-ef1b-4083-842f-65809d5d188c',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '43cf2738-5b7d-4a9b-9695-773699dc7704',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '76dd744d-8f86-4234-aa2b-fd4fb86f4f4e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e816a57a-6ee0-4b20-bd9a-8658c9a785ff',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '673021d7-5d77-4157-b623-e9fec65ed5d2',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '07d13a26-dd1b-4f28-b112-efaa501b1e53',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '514dc03a-881c-4caf-8e32-855371e6cceb',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.370077
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 1.109378
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 0.767345
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 0.557979
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.476149
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.332607
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.375723
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.306516
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.304072
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.254565
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.290967
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.228900
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.245962
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.245583
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.149385
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.120159
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.227544
Train Epoch: 1 [43520/60000 images (73%)]